In [2]:
import pandas as pd
import json
import re

# ── Load data ──────────────────────────────────────────────────────────────────
with open('results_with_gt.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# ── Clean HTML tags (qwen35 outputs a lot of HTML) ─────────────────────────────
def clean_text(text):
    if not text or not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'```html|```', '', text)
    return text.strip()

for col in ['ocr_qwen35', 'ocr_qwen3vl8b', 'ocr_firered', 'ground_truth']:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# ── Output length ───────────────────────────────────────────────────────────────
df['gt_len']         = df['ground_truth'].apply(len)
df['qwen35_len']     = df['ocr_qwen35'].apply(len)
df['qwen3vl8b_len']  = df['ocr_qwen3vl8b'].apply(len)
df['firered_len']    = df['ocr_firered'].apply(len)

# ── Length ratio vs ground truth (1.0 = perfect, <1 = too short, >1 = too long)
df['qwen35_ratio']    = (df['qwen35_len']    / df['gt_len'].replace(0, 1)).round(2)
df['qwen3vl8b_ratio'] = (df['qwen3vl8b_len'] / df['gt_len'].replace(0, 1)).round(2)
df['firered_ratio']   = (df['firered_len']   / df['gt_len'].replace(0, 1)).round(2)

# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("1. AVERAGE OUTPUT LENGTH BY LANGUAGE")
print("=" * 60)
print(df.groupby('language')[['gt_len', 'qwen35_len', 'qwen3vl8b_len', 'firered_len']].mean().round(0))

print("\n" + "=" * 60)
print("2. AVERAGE OUTPUT LENGTH BY LAYOUT")
print("=" * 60)
print(df.groupby('layout')[['gt_len', 'qwen35_len', 'qwen3vl8b_len', 'firered_len']].mean().round(0))

print("\n" + "=" * 60)
print("3. OVERALL STATISTICS")
print("=" * 60)
print(df[['gt_len', 'qwen35_len', 'qwen3vl8b_len', 'firered_len']].describe().round(0))

print("\n" + "=" * 60)
print("4. LENGTH RATIO VS GROUND TRUTH (1.0 = perfect)")
print("=" * 60)
print(df[['image', 'language', 'layout', 'qwen35_ratio', 'qwen3vl8b_ratio', 'firered_ratio']].to_string(index=False))

print("\n" + "=" * 60)
print("5. EMPTY OUTPUTS PER MODEL")
print("=" * 60)
for col in ['ocr_qwen35', 'ocr_qwen3vl8b', 'ocr_firered']:
    empty = df[df[col] == '']['image'].tolist()
    print(f"\n{col}: {len(empty)} empty output(s)")
    for img in empty:
        print(f"  - {img[:60]}")

print("\n" + "=" * 60)
print("6. SAMPLE COMPARISON (first 3 images)")
print("=" * 60)
for _, row in df.head(3).iterrows():
    print(f"\nImage   : {row['image'][:55]}")
    print(f"Language: {row['language']}  |  Layout: {row['layout']}")
    print(f"GT      : {row['ground_truth'][:120]}...")
    print(f"Qwen35  : {row['ocr_qwen35'][:120]}...")
    print(f"VL-8B   : {row['ocr_qwen3vl8b'][:120]}...")
    print(f"FireRed : {row['ocr_firered'][:120]}...")
    print("-" * 60)

# ── Save cleaned dataframe ──────────────────────────────────────────────────────
df.to_csv('results_cleaned.csv', index=False)
print("\nSaved cleaned results to results_cleaned.csv")


1. AVERAGE OUTPUT LENGTH BY LANGUAGE
          gt_len  qwen35_len  qwen3vl8b_len  firered_len
language                                                
de         210.0       216.0          216.0        187.0
en         452.0       483.0          477.0        471.0
fr         576.0       573.0          555.0        566.0
it        1335.0      1149.0          383.0       1179.0
la         129.0       263.0          213.0        226.0
nl        2305.0      2290.0         1323.0       2274.0

2. AVERAGE OUTPUT LENGTH BY LAYOUT
                       gt_len  qwen35_len  qwen3vl8b_len  firered_len
layout                                                               
image_with_caption      318.0       357.0          337.0        331.0
mixed_complex          2082.0      2056.0          409.0       2104.0
single_column          2173.0      2051.0         2020.0       2030.0
text_heavy_structured  1076.0       828.0          604.0        828.0

3. OVERALL STATISTICS
       gt_len  qwen35_len  q

In [3]:
import pandas as pd

df = pd.read_csv('/Users/miffycheng/uiuc course material/Research/exp/results_cleaned.csv')

# 排除沒有 GT 的圖片
df_valid = df[df['gt_len'] > 0].copy()

# 整體平均
print("Overall avg ratio:")
print(df_valid[['qwen35_ratio','qwen3vl8b_ratio','firered_ratio']].mean().round(2))

Overall avg ratio:
qwen35_ratio       0.99
qwen3vl8b_ratio    0.67
firered_ratio      0.91
dtype: float64


In [4]:
import pandas as pd

df = pd.read_csv('/Users/miffycheng/uiuc course material/Research/exp/results_cleaned.csv')
df_valid = df[df['gt_len'] > 0].copy()

print("=== Overall ===")
print(df_valid[['qwen35_ratio','qwen3vl8b_ratio','firered_ratio']].mean().round(2))

print("\n=== By Layout ===")
print(df_valid.groupby('layout')[['qwen35_ratio','qwen3vl8b_ratio','firered_ratio']].mean().round(2))

print("\n=== By Language ===")
print(df_valid.groupby('language')[['qwen35_ratio','qwen3vl8b_ratio','firered_ratio']].mean().round(2))

=== Overall ===
qwen35_ratio       0.99
qwen3vl8b_ratio    0.67
firered_ratio      0.91
dtype: float64

=== By Layout ===
                       qwen35_ratio  qwen3vl8b_ratio  firered_ratio
layout                                                             
image_with_caption             1.21             0.99           0.98
mixed_complex                  0.98             0.40           0.99
single_column                  0.82             0.78           0.84
text_heavy_structured          0.74             0.49           0.74

=== By Language ===
          qwen35_ratio  qwen3vl8b_ratio  firered_ratio
language                                              
de                1.03             1.03           0.89
en                1.04             1.00           0.98
fr                0.99             0.96           0.98
it                0.81             0.41           0.70
la                2.04             1.65           1.75
nl                0.99             0.50           0.99
